# D — Modeling & Evaluation

**Forecasting rule:** chronological validation only. The last 14 days in the training period are held out; the November test set is never used to calculate scores. Operational columns (`avg_fare_birr`, `avg_wait_min`, `active_drivers`) are excluded because they are not known at forecast time.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from pipeline import run_pipeline
result=run_pipeline(run_models=True)
print('Pipeline/model run complete:',result['train'].shape,result['test'].shape)

## D1–D2 Baselines and model comparison

In [ ]:
comparison=pd.read_csv(ROOT/'reports/D_model_comparison.csv'); display(comparison.sort_values('RMSE').round(4))
print('Winner is the model with the lowest RMSE on the same chronological validation window; also inspect MAE and training time.')

## D3 Rolling-origin validation

In [ ]:
rolling=pd.read_csv(ROOT/'reports/D_rolling_origin.csv'); display(rolling)
print(rolling.groupby('model').RMSE.agg(['mean','std','count']).round(4))

## D4 Feature availability and leakage audit

In [ ]:
display(pd.read_csv(ROOT/'reports/D4_feature_availability_audit.csv'))
print('Excluded train-only columns: avg_fare_birr, avg_wait_min, active_drivers. Never score the test file.')

## D5 Ablation

In [ ]:
display(pd.read_csv(ROOT/'reports/D_ablation.csv').round(4))
print('Compare calendar+zone+trend, plus weather, plus events, and both. Negative/weak gains are still useful evidence and should be reported honestly.')

## D6–D8 Tuning, error analysis, response to findings

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT/'src'/'tune_model.py')], check=True)
display(pd.read_csv(ROOT/'reports/D6_tuning.csv'))
for name in ['D_error_by_zone.csv','D_error_by_hour.csv','D_largest_errors.csv']:
 p=ROOT/'reports'/name
 if p.exists():
  print('\n',name); display(pd.read_csv(p).head(12))
print('The best trial is selected using chronological validation only; no hidden test labels are used.')

## D9 Plain-language interpretation and submission QA

In [ ]:
sub=pd.read_csv(ROOT/'submission/team_ride_demand_submission.csv'); test=pd.read_csv(ROOT/'data/raw/ride_demand_test.csv'); print('Submission shape:',sub.shape,'columns:',sub.columns.tolist()); print('IDs/order match:',sub.row_id.equals(test.row_id)); print('Missing:',sub.predicted_trips.isna().sum(),'negative:',(sub.predicted_trips<0).sum()); print('Mean predicted trips:',round(sub.predicted_trips.mean(),2))
print('Operations translation: RMSE/MAE are in trips per zone-hour. Approximate driver requirement = forecast trips / 1.3 trips per driver-hour; gross fare estimate uses historical zone average fare in the app.')